In [ ]:
import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.layers import Conv1D, Dense, Flatten, Dropout, MaxPooling1D, BatchNormalization
from tensorflow.keras.models import Sequential
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
import tensorflow_datasets as tfds
import keras_tuner as kt
from larq.layers import QuantConv1D, QuantDense
from larq.models import summary
import larq_compute_engine as lce
import random
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, precision_recall_fscore_support
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.utils import shuffle
import numpy as np
import scipy.io
import joblib
import numpy as np
import pandas as pd
from tabulate import tabulate

In [ ]:
# Check if GPU is Available
print("Num GPUs Available: ", tf.config.list_physical_devices('GPU'))
gpus = tf.config.experimental.list_physical_devices('GPU')
if gpus:
    # Print information about each GPU
    for gpu in gpus:
        print("GPU:", gpu.name)
else:
    print("No GPU devices found.")

## Dataset Preparation

In [ ]:
def load_speech_commands_dataset(window_length=16000, sample_rate=16000, val_split=0.1, test_split=0.1, classes=None, seed=42):
    """
    Load the Speech Commands dataset dynamically using TensorFlow Datasets.
    
    Args:
        window_length: Length of audio window (default 16000 samples = 1 second at 16kHz)
        sample_rate: Audio sample rate
        val_split: Validation split ratio
        test_split: Test split ratio
        classes: List of classes to include (None means all classes)
        seed: Random seed for reproducibility
        
    Returns:
        X_train, X_val, X_test, y_train, y_val, y_test, width
    """
    
    # Set random seeds for reproducibility
    random.seed(seed)
    tf.random.set_seed(seed)
    np.random.seed(seed)
    
    # Load the Speech Commands dataset
    print("Loading Speech Commands dataset...")
    dataset, info = tfds.load('speech_commands', with_info=True)
    
    # Get all available classes
    all_labels = info.features['label'].names
    print(f"Available classes: {all_labels}")
    
    # Filter classes if specified
    if classes is not None:
        class_indices = [all_labels.index(c) for c in classes if c in all_labels]
    else:
        classes = all_labels
        class_indices = list(range(len(all_labels)))
    
    print(f"Using classes: {[all_labels[i] for i in class_indices]}")
    
    def preprocess(example):
        """Preprocess a single example from the dataset."""
        audio = example['audio']
        label = example['label']
        
        # Resample if needed
        if example['audio'].shape[0] != sample_rate:
            audio = tf.audio.resample(audio, tf.cast(example['audio'].shape[0], tf.int64), sample_rate)
        
        # Pad or truncate to window_length
        if audio.shape[0] < window_length:
            padding = window_length - audio.shape[0]
            audio = tf.pad(audio, [[0, padding]])
        else:
            audio = audio[:window_length]
        
        # Normalize the audio
        audio = tf.cast(audio, tf.float32) / 32768.0  # Convert to float and normalize to [-1, 1]
        
        # Reshape for model input (add channel dimension)
        audio = tf.reshape(audio, [window_length, 1])
        
        # Filter by selected classes
        keep_sample = tf.reduce_any(tf.equal(label, class_indices))
        
        # Map label to new index range
        new_label = tf.cast(tf.where(tf.equal(label, class_indices))[0][0], tf.float32)
        
        return audio, new_label, keep_sample
    
    # Apply preprocessing to training set
    dataset_processed = dataset['train'].map(preprocess)
    
    # Filter to keep only samples from selected classes
    dataset_filtered = dataset_processed.filter(lambda _, __, keep: keep)
    
    # Remove the keep_sample field
    dataset_filtered = dataset_filtered.map(lambda audio, label, _: (audio, label))
    
    # Get total number of examples
    total_examples = dataset_filtered.reduce(0, lambda x, _: x + 1).numpy()
    print(f"Total filtered examples: {total_examples}")
    
    # Shuffle dataset
    dataset_filtered = dataset_filtered.shuffle(buffer_size=10000, seed=seed)
    
    # Calculate split sizes
    val_size = int(total_examples * val_split)
    test_size = int(total_examples * test_split)
    train_size = total_examples - val_size - test_size
    
    # Split dataset
    train_ds = dataset_filtered.take(train_size)
    temp_ds = dataset_filtered.skip(train_size)
    val_ds = temp_ds.take(val_size)
    test_ds = temp_ds.skip(val_size)
    
    # Function to convert dataset to tensors
    def dataset_to_tensors(ds):
        audios = []
        labels = []
        for audio, label in ds:
            audios.append(audio.numpy())
            labels.append(label.numpy())
        
        X = np.array(audios, dtype=np.float32)
        y = np.array(labels, dtype=np.float32)
        return X, y
    
    # Convert to tensors
    print("Converting to tensors...")
    X_train, y_train = dataset_to_tensors(train_ds)
    X_val, y_val = dataset_to_tensors(val_ds)
    X_test, y_test = dataset_to_tensors(test_ds)
    
    print(f"Train set: {X_train.shape}, Validation set: {X_val.shape}, Test set: {X_test.shape}")
    
    # Convert to TensorFlow tensors
    X_train = tf.convert_to_tensor(X_train, dtype=tf.float32)
    y_train = tf.convert_to_tensor(y_train, dtype=tf.float32)
    X_val = tf.convert_to_tensor(X_val, dtype=tf.float32)
    y_val = tf.convert_to_tensor(y_val, dtype=tf.float32)
    X_test = tf.convert_to_tensor(X_test, dtype=tf.float32)
    y_test = tf.convert_to_tensor(y_test, dtype=tf.float32)
    
    width = window_length
    
    return X_train, X_val, X_test, y_train, y_val, y_test, width

In [ ]:
classes = ['01','03','04','05', '06','07','09','10','11', '12','13','14','15','16','17']
X_train, X_val, X_test, y_train, y_val, y_test, width = dataset(classes, val=True, window_length=65536)

## NN Evaluation Methods

In [ ]:
def evaluate_NN_models(model_list, X_test, y_test, bin_class=False):
    loss, accuracy, y_pred, precision, recall, f1_score, support, confusion_matrix_list = [], [], [], [], [], [], [], []
    for model in model_list:
        loss_value, accuracy_value = model.evaluate(X_test, y_test)
        loss.append(loss_value)
        accuracy.append(accuracy_value)
        if bin_class:
            y_pred_value = (model.predict(X_test) > 0.5).astype(int)
        else:
            y_pred_value = np.argmax(model.predict(X_test), axis=1)
        y_pred.append(y_pred_value)
        y_test_classes = y_test
        precision_value, recall_value, f1_score_value, support_value = precision_recall_fscore_support(
            y_test_classes, 
            y_pred_value, 
            average='binary' if bin_class else 'weighted',
            zero_division=0
        )
        precision.append(precision_value)
        recall.append(recall_value)
        f1_score.append(f1_score_value)
        support.append(support_value)
        confusion_matrix_value = confusion_matrix(y_test_classes, y_pred_value)
        confusion_matrix_list.append(confusion_matrix_value)
    return [accuracy, precision, recall, f1_score], confusion_matrix_list, y_pred

In [ ]:
def print_table(data,headers):
    table_data = list(zip(*data))
    table = tabulate(table_data, headers=headers, tablefmt='grid')
    print(table)

In [ ]:
def plot_confusion_matrix(ax, conf_matrix, title, cmap):
    sns.heatmap(conf_matrix, annot=True, fmt='d', cmap=cmap, cbar=False, ax=ax)
    ax.set_title(title)

In [ ]:
def draw_confusion_matrix(matrix,name):
    colormap=["Blues","Greens","Oranges"]
    fig, axes = plt.subplots(1, 1, figsize=(15, 5))
    for i in range(len(matrix)):
        plot_confusion_matrix(axes, matrix[i], "Confusion Matrix  ", colormap[i])
    plt.tight_layout()

## Hyperparameters Optimization

In [ ]:
class BinaryNetHyperModel(kt.HyperModel):
    def __init__(self, input_shape, num_classes):
        self.input_shape = input_shape
        self.num_classes = num_classes

    def build(self, hp):
        kwargs = dict(input_quantizer="ste_sign",
                     kernel_quantizer="ste_sign",
                     kernel_constraint="weight_clip")

        model = Sequential()
        
        # Tune the first Conv1D layer
        filters_conv1 = hp.Int('conv1_filters', min_value=16, max_value=64, step=16)
        model.add(Conv1D(filters=filters_conv1, kernel_size=3, activation='relu', 
                        input_shape=self.input_shape))
        model.add(MaxPooling1D(pool_size=2))
        model.add(BatchNormalization(scale=False))

        # Tune the first QuantConv1D layer
        filters_qconv1 = hp.Int('qconv1_filters', min_value=4, max_value=16, step=4)
        model.add(QuantConv1D(filters_qconv1, kernel_size=3,
                             kernel_quantizer="ste_sign",
                             kernel_constraint="weight_clip",
                             use_bias=False))
        model.add(MaxPooling1D(pool_size=2))
        model.add(BatchNormalization(scale=False))
        model.add(Dropout(rate=hp.Float('dropout1', min_value=0.2, max_value=0.5)))

        # Tune the second QuantConv1D layer
        filters_qconv2 = hp.Int('qconv2_filters', min_value=4, max_value=16, step=4)
        model.add(QuantConv1D(filters_qconv2, kernel_size=3, use_bias=False, **kwargs))
        model.add(MaxPooling1D(pool_size=2))
        model.add(BatchNormalization(scale=False))
        model.add(Dropout(rate=hp.Float('dropout2', min_value=0.2, max_value=0.5)))

        model.add(Flatten())

        # Tune the QuantDense layer
        units_qdense = hp.Int('qdense_units', min_value=4, max_value=16, step=4)
        model.add(QuantDense(units_qdense, use_bias=False, **kwargs))
        model.add(BatchNormalization(scale=False))
        model.add(Dense(self.num_classes, activation='softmax'))

        # Tune learning rate
        learning_rate = hp.Float('learning_rate', min_value=1e-4, max_value=1e-2, sampling='log')
        model.compile(
            optimizer=Adam(learning_rate=learning_rate),
            loss='sparse_categorical_crossentropy',
            metrics=['accuracy']
        )
        return model

In [ ]:
# Initialize the hypermodel
hypermodel = BinaryNetHyperModel(input_shape=(width, 1), num_classes=len(classes))

# Create the tuner
tuner = kt.BayesianOptimization(
    hypermodel,
    objective='val_accuracy',
    max_trials=20,
    directory='bayesian_opt',
    project_name='z24_tuning'
)

# Define early stopping callback
early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=50,
    verbose=1,
    mode='min',
    restore_best_weights=True
)

# Search for the best hyperparameters
tuner.search(
    X_train,
    y_train,
    epochs=100,
    batch_size=64,
    validation_data=(X_val, y_val),
    callbacks=[early_stopping]
)

# Get the best hyperparameters
best_hps_list = tuner.get_best_hyperparameters(num_trials=3)

In [ ]:
# Build and evaluate the top 3 models
models = []
scores = []

print("\nTop 3 Models Performance:")
print("-" * 50)

for idx, hps in enumerate(best_hps_list, 1):
    print(f"\nModel {idx}:")
    print("Hyperparameters:")
    for param, value in hps.values.items():
        print(f"{param}: {value}")
            
    # Build and evaluate the model
    model = tuner.hypermodel.build(hps)
    model.fit(X_train, y_train, 
             epochs=10,  # Quick evaluation
             batch_size=32,
             validation_data=(X_val, y_val),
             verbose=0)
    
    val_loss, val_acc = model.evaluate(X_val, y_val, verbose=0)
    print(f"Validation Accuracy: {val_acc:.4f}")
    
    models.append(model)
    scores.append(val_acc)
        
# Let user choose the model
while True:
    try:
        choice = int(input("\nChoose a model (1, 2, or 3): "))
        if 1 <= choice <= 3:
            break
        print("Please enter 1, 2, or 3")
    except ValueError:
        print("Please enter a valid number")

# Use the chosen model
model = models[choice - 1]
chosen_hps = best_hps_list[choice - 1]

print(f"\nSelected Model {choice}")
print("Final hyperparameters:")
for param, value in chosen_hps.values.items():
    print(f"{param}: {value}")

## Build BNN

In [ ]:
# All quantized layers except the first will use the same options
kwargs = dict(input_quantizer="ste_sign",
            kernel_quantizer="ste_sign",
            kernel_constraint="weight_clip")

model = Sequential()
model.add(Conv1D(filters=32, kernel_size=3, activation='relu', input_shape=(width,1)))
model.add(MaxPooling1D(pool_size=2))
# In the first layer we only quantize the weights and not the input
model.add(QuantConv1D(8, kernel_size=3,
                                kernel_quantizer="ste_sign",
                                kernel_constraint="weight_clip",
                                use_bias=False))
model.add(MaxPooling1D(pool_size=2))
model.add(Dropout(rate=0.5))

model.add(QuantConv1D(8, kernel_size=3, use_bias=False, **kwargs))
model.add(MaxPooling1D(pool_size=2))
model.add(Dropout(rate=0.5))

model.add(Flatten())

model.add(QuantDense(8, use_bias=False, **kwargs))
model.add(BatchNormalization(scale=False))
model.add(Dense(len(classes), activation='softmax'))

In [ ]:
# All quantized layers except the first will use the same options
kwargs = dict(input_quantizer="ste_sign",
            kernel_quantizer="ste_sign",
            kernel_constraint="weight_clip")

model = Sequential()
# First Conv1D layer with tuned filters
model.add(Conv1D(filters=chosen_hps.get('conv1_filters'), kernel_size=3, activation='relu', 
                 input_shape=(width,1)))
model.add(MaxPooling1D(pool_size=2))
model.add(BatchNormalization(scale=False))

# First QuantConv1D layer with tuned filters
model.add(QuantConv1D(chosen_hps.get('qconv1_filters'), kernel_size=3,
                      kernel_quantizer="ste_sign",
                      kernel_constraint="weight_clip",
                      use_bias=False))
model.add(MaxPooling1D(pool_size=2))
model.add(BatchNormalization(scale=False))
model.add(Dropout(rate=chosen_hps.get('dropout1')))

# Second QuantConv1D layer with tuned filters
model.add(QuantConv1D(chosen_hps.get('qconv2_filters'), kernel_size=3, use_bias=False, **kwargs))
model.add(MaxPooling1D(pool_size=2))
model.add(BatchNormalization(scale=False))
model.add(Dropout(rate=chosen_hps.get('dropout2')))

model.add(Flatten())

# QuantDense layer with tuned units
model.add(QuantDense(chosen_hps.get('qdense_units'), use_bias=False, **kwargs))
model.add(BatchNormalization(scale=False))
model.add(Dense(len(classes), activation='softmax'))

In [ ]:
lr = chosen_hps.get('learning_rate')
epochs = 300
batch_size = 64

In [ ]:
model.compile(optimizer=Adam(learning_rate=lr), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
print(summary(model))

### Train

In [ ]:
history= model.fit(X_train, y_train, epochs=epochs, batch_size=batch_size, validation_data=(X_val, y_val), callbacks=[early_stopping], verbose=1)

### Evaluate

In [ ]:
result_all_models_FC, confusion_matrix_FC, predicitions_FC = evaluate_NN_models([model], X_test, y_test, bin_class=False)

In [ ]:
headers=['models','accuracy','precision','recall','f1_score']
data_normal_FC=[["model"]]
data_normal_FC.extend(result_all_models_FC)
print_table(data_normal_FC,headers)

In [ ]:
draw_confusion_matrix(confusion_matrix_FC,"Confusion Matrix")

In [ ]:
plt.figure()
# plt.plot(history.history['accuracy'], 'g', label='Training Accuracy')
plt.plot(history.history['val_accuracy'], color='orange', label='Validation Accuracy')
plt.xlabel("epochs")
plt.ylabel("accuracy")
plt.show()

## Save

In [ ]:
# Convert our Keras model to a TFLite flatbuffer file
with open("z24.tflite", "wb") as flatbuffer_file:
    flatbuffer_bytes = lce.convert_keras_model(model)
    flatbuffer_file.write(flatbuffer_bytes)